# NVIDIA Cosmos — Local Video & Image Q&A

Interactive multi-turn visual question-answering using `nvidia/Cosmos-Reason2-2B` loaded locally via Hugging Face Transformers.

- Supports **video** (`mp4`, `avi`, etc.) and **image** (`jpg`, `png`, `webp`, etc.) inputs
- Media is attached **once** on the first user turn; follow-up questions are text-only
- Reasoning traces (`<think>...</think>`) are parsed and displayed separately

## 1. Imports & Model Loading

In [ ]:
import re
import torch
import transformers

MODEL_NAME = "nvidia/Cosmos-Reason2-2B"
MAX_NEW_TOKENS = 4096
VIDEO_FPS = 4

print(f"Loading model: {MODEL_NAME}")
model = transformers.Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    attn_implementation="sdpa",
)
processor: transformers.Qwen3VLProcessor = transformers.AutoProcessor.from_pretrained(MODEL_NAME)
print(f"Model loaded on: {model.device}")

## 2. Configuration — Set Your Media Path Here

Point `MEDIA_PATH` to either a **video** or an **image** file. The media type is detected automatically from the file extension.

In [ ]:
import pathlib

VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv", ".webm", ".flv", ".wmv"}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif", ".tiff", ".tif"}


def detect_media_type(path: str) -> str:
    """Return 'video', 'image', or raise ValueError for unsupported extensions."""
    ext = pathlib.Path(path).suffix.lower()
    if ext in VIDEO_EXTENSIONS:
        return "video"
    if ext in IMAGE_EXTENSIONS:
        return "image"
    raise ValueError(f"Unsupported file extension '{ext}'. Add it to VIDEO_EXTENSIONS or IMAGE_EXTENSIONS.")


# ── Set this to your video or image file path ──────────────────────────────
MEDIA_PATH = "/path/to/your/file.mp4"
# ───────────────────────────────────────────────────────────────────────────

media_uri = MEDIA_PATH  # processor accepts plain local paths directly
media_type = detect_media_type(MEDIA_PATH)
print(f"Media type : {media_type}")
print(f"Media path : {media_uri}")

## 3. Inference Helpers

In [ ]:
import cv2
from PIL import Image

SYSTEM_MESSAGE = {
    "role": "system",
    "content": [{"type": "text", "text": "You are a helpful assistant."}],
}


def build_media_content(uri: str, kind: str) -> dict:
    """Return the content dict for a video or image attachment."""
    if kind == "video":
        return {"type": "video", "video": uri, "fps": VIDEO_FPS}
    return {"type": "image", "image": uri}


def build_user_message(text: str, media_uri: str | None = None, media_type: str | None = None) -> dict:
    """Build a user turn. Attach media only when media_uri is provided."""
    content = []
    if media_uri and media_type:
        content.append(build_media_content(media_uri, media_type))
    content.append({"type": "text", "text": text})
    return {"role": "user", "content": content}


def decode_video(path: str, fps: int = VIDEO_FPS) -> list:
    """Decode a video file to a list of PIL RGB frames sampled at `fps`."""
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video: {path}")
    video_fps = cap.get(cv2.CAP_PROP_FPS) or fps
    frame_interval = max(1, round(video_fps / fps))
    frames = []
    idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % frame_interval == 0:
            frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
        idx += 1
    cap.release()
    if not frames:
        raise ValueError(f"No frames extracted from: {path}")
    print(f"  Decoded {len(frames)} frames from video.")
    return frames


def extract_media(messages: list) -> tuple[list, list]:
    """Walk all messages and collect video/image paths in order."""
    videos, images = [], []
    for msg in messages:
        content = msg.get("content", [])
        if isinstance(content, list):
            for item in content:
                if not isinstance(item, dict):
                    continue
                if item.get("type") == "video":
                    videos.append(item["video"])
                elif item.get("type") == "image":
                    images.append(item["image"])
    return videos, images


def run_inference(history: list) -> str:
    """Run the model on the current conversation history and return the assistant reply."""
    messages = [SYSTEM_MESSAGE] + history

    # Render the chat template to text only
    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # Pre-decode media using cv2/PIL so we never rely on transformers'
    # fetch_videos path (which requires torchcodec or a newer torchvision)
    video_paths, image_paths = extract_media(messages)
    decoded_videos = [decode_video(p) for p in video_paths] if video_paths else None
    decoded_images = [Image.open(p).convert("RGB") for p in image_paths] if image_paths else None

    inputs = processor(
        text=[text],
        videos=decoded_videos,
        images=decoded_images,
        return_tensors="pt",
    )
    inputs = inputs.to(model.device)

    generated_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)
    trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids, strict=False)
    ]
    output = processor.batch_decode(
        trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )
    return output[0]


def parse_response(raw: str) -> tuple[str, str]:
    """Split <think>...</think> reasoning from the final answer."""
    match = re.search(r"<think>(.*?)</think>(.*)", raw, re.DOTALL)
    if match:
        return match.group(1).strip(), match.group(2).strip()
    return "", raw.strip()


def display_response(raw: str) -> None:
    thinking, answer = parse_response(raw)
    if thinking:
        print("=" * 60)
        print("THINKING")
        print("=" * 60)
        print(thinking)
        print()
    print("=" * 60)
    print("ANSWER")
    print("=" * 60)
    print(answer)
    print()


print("Helpers defined.")

## 4. Single-Turn Example

Run one question against the loaded media — good for quick testing.

Edit `question` to match your use case (the default is phrased for a driving scene).

In [ ]:
question = (
    "Describe what you see. Answer using the following format:\n\n"
    "<think>\nYour reasoning.\n</think>\n\n"
    "Write your final answer immediately after the </think> tag."
)

history = [build_user_message(question, media_uri=media_uri, media_type=media_type)]
raw = run_inference(history)
display_response(raw)

## 5. Multi-Turn Interactive Session

The media is attached only on the first user message; follow-up questions are text-only.
Type `quit` or press `Ctrl-C` to stop.

In [ ]:
print(f"NVIDIA Cosmos — {MODEL_NAME}")
print("=" * 60)
print(f"{media_type.capitalize()}: {media_uri}")
print(f"Ask questions about the {media_type}. Type 'quit' or Ctrl-C to stop.\n")

history: list = []

while True:
    try:
        question = input("You: ").strip()
    except (KeyboardInterrupt, EOFError):
        print("\nExiting.")
        break

    if not question:
        continue
    if question.lower() in {"quit", "exit", "q"}:
        print("Exiting.")
        break

    # Attach media only on the first user turn
    user_msg = build_user_message(
        question,
        media_uri=media_uri if not history else None,
        media_type=media_type if not history else None,
    )
    history.append(user_msg)

    print("\nModel: ", end="", flush=True)
    raw = run_inference(history)
    display_response(raw)

    _, answer = parse_response(raw)
    history.append({"role": "assistant", "content": answer})

## 6. Inspect Conversation History

In [ ]:
import json
print(json.dumps(history, indent=2, default=str))